# LeetCode 2053 — Kth Distinct String in an Array bằng Python

Link đề: https://leetcode.com/problems/kth-distinct-string-in-an-array/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, String, Counting

```python
class Solution:
    def kthDistinct(self, arr: List[str], k: int) -> str:

```

**Mục tiêu học**

1. Hiểu đúng thuật ngữ "**distinct string**" của đề: chuỗi xuất hiện **đúng một lần** (không phải "các giá trị khác nhau" như `set`).
2. Thành thạo kỹ thuật **đếm tần suất hai lượt** (two-pass counting): lượt 1 đếm, lượt 2 duyệt theo thứ tự gốc.
3. Dùng `dict`, `dict.get`, `collections.Counter`, và tính chất **dict giữ thứ tự chèn** (Python 3.7+).
4. So sánh brute force `arr.count(x)` `O(n²)` với cách `O(n)`.
5. Nộp `class Solution`, xử lý đúng trường hợp "ít hơn k chuỗi distinct" → trả `""`.

## 1. Đề bài yêu cầu gì?

Một **distinct string** là chuỗi xuất hiện **đúng một lần** trong mảng. Cho mảng chuỗi `arr` và số nguyên `k`, trả về chuỗi distinct **thứ k** (đếm theo **thứ tự xuất hiện** trong mảng). Nếu có ít hơn `k` chuỗi distinct, trả `""`.

| Ví dụ | arr | k | Output | Vì sao |
|---|---|---|---|---|
| 1 | `["d","b","c","b","c","a"]` | 2 | `"a"` | distinct theo thứ tự: `d`, `a` → thứ 2 là `a` |
| 2 | `["aaa","aa","a"]` | 1 | `"aaa"` | cả 3 đều distinct |
| 3 | `["a","b","a"]` | 3 | `""` | chỉ có `b` là distinct |

Constraints: `1 <= k <= arr.length <= 1000`, `1 <= arr[i].length <= 5`, chỉ chữ thường.

**Bẫy thuật ngữ:** trong tiếng Anh thông thường "distinct values" = các giá trị khác nhau (như `set`). Nhưng đề này **định nghĩa lại**: distinct = xuất hiện đúng 1 lần. Ở ví dụ 1, `set(arr)` có 4 giá trị nhưng chỉ có 2 chuỗi distinct.

In [1]:
arr = ["d", "b", "c", "b", "c", "a"]
print("set(arr)            :", sorted(set(arr)), "-> 4 giá trị khác nhau")
print("xuất hiện đúng 1 lần:", [x for x in arr if arr.count(x) == 1])

set(arr)            : ['a', 'b', 'c', 'd'] -> 4 giá trị khác nhau
xuất hiện đúng 1 lần: ['d', 'a']


## 2. Nền tảng Python

### 2.1. Chỉ số k bắt đầu từ 1

Đề đếm "thứ 1, thứ 2, ..." (1-indexed) còn list Python đánh số từ 0. Distinct thứ `k` là phần tử `distinct_list[k - 1]`.

In [2]:
distinct_list = ["d", "a"]
k = 2
print(distinct_list[k - 1])     # 'a'
k = 3
print(k <= len(distinct_list))  # False -> phải trả ""

a
False


### 2.2. `list.count(x)` — tiện nhưng đắt

`arr.count(x)` duyệt **toàn bộ** list để đếm → `O(n)` mỗi lần gọi. Gọi trong vòng lặp `n` lần → `O(n²)`.

In [3]:
arr = ["d", "b", "c", "b", "c", "a"]
for x in arr:
    print(x, "xuất hiện", arr.count(x), "lần")

d xuất hiện 1 lần
b xuất hiện 2 lần
c xuất hiện 2 lần
b xuất hiện 2 lần
c xuất hiện 2 lần
a xuất hiện 1 lần


### 2.3. `dict` đếm tần suất và `dict.get`

`freq[x] = freq.get(x, 0) + 1`: nếu `x` chưa có thì `get` trả `0` → gán 1; nếu có rồi thì cộng thêm. Tra/cập nhật `dict` là `O(1)` trung bình (dựa trên băm — hash).

In [4]:
freq = {}
for x in ["d", "b", "c", "b", "c", "a"]:
    freq[x] = freq.get(x, 0) + 1
    print(f"thêm {x!r}: freq = {freq}")

thêm 'd': freq = {'d': 1}
thêm 'b': freq = {'d': 1, 'b': 1}
thêm 'c': freq = {'d': 1, 'b': 1, 'c': 1}
thêm 'b': freq = {'d': 1, 'b': 2, 'c': 1}
thêm 'c': freq = {'d': 1, 'b': 2, 'c': 2}
thêm 'a': freq = {'d': 1, 'b': 2, 'c': 2, 'a': 1}


### 2.4. `Counter` — đếm sẵn trong một dòng

`Counter(arr)` trả về một dict đặc biệt: key là phần tử, value là số lần. Truy cập key không có trả `0` (không báo lỗi).

In [5]:
from collections import Counter
cnt = Counter(["d", "b", "c", "b", "c", "a"])
print(cnt)
print(cnt["b"], cnt["zzz"])      # 2 0

Counter({'b': 2, 'c': 2, 'd': 1, 'a': 1})
2 0


### 2.5. Dict giữ **thứ tự chèn**

Từ Python 3.7, khi duyệt một `dict` (và `Counter`), các key ra theo **thứ tự lần đầu được thêm vào**. Vì vậy duyệt `freq` cũng chính là duyệt các giá trị theo thứ tự xuất hiện **lần đầu** trong `arr` — rất hợp với bài này (chuỗi distinct chỉ xuất hiện một lần, nên lần đầu = lần duy nhất).

In [6]:
freq = {}
for x in ["z", "a", "m", "a"]:
    freq[x] = freq.get(x, 0) + 1
print(list(freq))           # ['z', 'a', 'm'] — đúng thứ tự gặp lần đầu, không phải bảng chữ cái

['z', 'a', 'm']


## 3. Cách 1 — Brute force: đếm bằng `arr.count`

Duyệt `arr` theo thứ tự; với mỗi `x`, nếu `arr.count(x) == 1` thì đó là một chuỗi distinct → giảm `k`; khi `k` về 0 thì trả `x`.

In [7]:
def kth_distinct_bruteforce(arr, k):
    for x in arr:
        if arr.count(x) == 1:
            k -= 1
            if k == 0:
                return x
    return ""

print(repr(kth_distinct_bruteforce(["d", "b", "c", "b", "c", "a"], 2)))   # 'a'
print(repr(kth_distinct_bruteforce(["aaa", "aa", "a"], 1)))               # 'aaa'
print(repr(kth_distinct_bruteforce(["a", "b", "a"], 3)))                  # ''

'a'
'aaa'
''


### Nút thắt

Mỗi `arr.count(x)` tốn `n` bước, lặp `n` lần → `n²`. Với `n = 1000` là 10⁶ phép so sánh chuỗi — vẫn qua LeetCode, nhưng ta đang **đếm lại cùng một chuỗi nhiều lần** (`"b"` được đếm 2 lần, mỗi lần duyệt cả mảng).

**Insight:** tần suất của mỗi chuỗi **không đổi** trong suốt bài. Hãy đếm **một lần cho tất cả** (lượt 1, `O(n)`), lưu vào dict, rồi tra `O(1)` (lượt 2). Đây là mẫu **two-pass counting**.

In [8]:
# Đo số phép so sánh của brute force để thấy n^2
def count_comparisons(n):
    arr = [str(i % (n // 2 + 1)) for i in range(n)]
    return sum(len(arr) for _ in arr)     # mỗi arr.count duyệt len(arr) phần tử
for n in [10, 100, 1000]:
    print(f"n={n:<5} brute force ~ {count_comparisons(n):>9} so sánh | two-pass ~ {2 * n}")

n=10    brute force ~       100 so sánh | two-pass ~ 20
n=100   brute force ~     10000 so sánh | two-pass ~ 200
n=1000  brute force ~   1000000 so sánh | two-pass ~ 2000


## 4. Cách 2 (lời giải chính) — Hai lượt: đếm rồi duyệt theo thứ tự gốc

- **Lượt 1:** `freq = Counter(arr)`.
- **Lượt 2:** duyệt `x` trong `arr` (giữ thứ tự gốc). Nếu `freq[x] == 1` → giảm `k`; `k == 0` → trả `x`.
- Hết mảng mà chưa trả → `""`.

In [9]:
def kth_distinct_two_pass(arr, k):
    freq = Counter(arr)                 # lượt 1
    for x in arr:                       # lượt 2: theo thứ tự xuất hiện
        if freq[x] == 1:
            k -= 1
            if k == 0:
                return x
    return ""

print(repr(kth_distinct_two_pass(["d", "b", "c", "b", "c", "a"], 2)))
print(repr(kth_distinct_two_pass(["aaa", "aa", "a"], 1)))
print(repr(kth_distinct_two_pass(["a", "b", "a"], 3)))

'a'
'aaa'
''


## 5. TRACE từng bước

Ví dụ 1: `arr = ["d","b","c","b","c","a"]`, `k = 2`. Lượt 1: `freq = {d:1, b:2, c:2, a:1}`.

| i | x | freq[x] | distinct? | k trước | k sau | hành động |
|---|---|---|---|---|---|---|
| 0 | d | 1 | ✓ | 2 | 1 | chưa đủ |
| 1 | b | 2 | ✗ | 1 | 1 | bỏ qua |
| 2 | c | 2 | ✗ | 1 | 1 | bỏ qua |
| 3 | b | 2 | ✗ | 1 | 1 | bỏ qua |
| 4 | c | 2 | ✗ | 1 | 1 | bỏ qua |
| 5 | a | 1 | ✓ | 1 | 0 | **trả "a"** |

In [10]:
def trace_kth_distinct(arr, k):
    freq = Counter(arr)
    print("Lượt 1 - freq:", dict(freq))
    for i, x in enumerate(arr):
        if freq[x] == 1:
            k -= 1
            print(f"i={i}, x={x!r}: freq=1 -> distinct, k còn {k}")
            if k == 0:
                print(f"k == 0 -> trả {x!r}")
                return x
        else:
            print(f"i={i}, x={x!r}: freq={freq[x]} -> bỏ qua")
    print("Hết mảng, chưa đủ k chuỗi distinct -> trả ''")
    return ""

trace_kth_distinct(["d", "b", "c", "b", "c", "a"], 2)

Lượt 1 - freq: {'d': 1, 'b': 2, 'c': 2, 'a': 1}
i=0, x='d': freq=1 -> distinct, k còn 1
i=1, x='b': freq=2 -> bỏ qua
i=2, x='c': freq=2 -> bỏ qua
i=3, x='b': freq=2 -> bỏ qua
i=4, x='c': freq=2 -> bỏ qua
i=5, x='a': freq=1 -> distinct, k còn 0
k == 0 -> trả 'a'


'a'

In [11]:
trace_kth_distinct(["a", "b", "a"], 3)

Lượt 1 - freq: {'a': 2, 'b': 1}
i=0, x='a': freq=2 -> bỏ qua
i=1, x='b': freq=1 -> distinct, k còn 2
i=2, x='a': freq=2 -> bỏ qua
Hết mảng, chưa đủ k chuỗi distinct -> trả ''


''

## 6. Cách 3 — Tạo danh sách distinct rồi lấy phần tử `k-1`

Viết bằng list comprehension, đọc rất "giống đề":

In [12]:
def kth_distinct_list(arr, k):
    freq = Counter(arr)
    distinct = [x for x in arr if freq[x] == 1]
    return distinct[k - 1] if k <= len(distinct) else ""

print(repr(kth_distinct_list(["d", "b", "c", "b", "c", "a"], 2)))
print(repr(kth_distinct_list(["a", "b", "a"], 3)))

'a'
''


`A if điều_kiện else B` là **biểu thức điều kiện** (ternary): trả `A` khi đúng, `B` khi sai. Cách này tốn thêm list `distinct` và không dừng sớm, nhưng vẫn `O(n)`.

## 7. Cách 4 — Một lượt với hai tập `seen` và `dup`, rồi duyệt dict

Không dùng Counter: duyệt một lượt, `seen` chứa chuỗi đã gặp, `dup` chứa chuỗi đã gặp **từ lần thứ 2** trở đi. Sau đó các chuỗi distinct = `seen - dup`, và để giữ thứ tự ta dùng một `dict` làm "set có thứ tự".

In [13]:
def kth_distinct_sets(arr, k):
    order = {}                 # dict làm "set có thứ tự": key = chuỗi, value bỏ qua
    dup = set()
    for x in arr:
        if x in order:
            dup.add(x)
        else:
            order[x] = True
    for x in order:            # thứ tự gặp lần đầu
        if x not in dup:
            k -= 1
            if k == 0:
                return x
    return ""

print(repr(kth_distinct_sets(["d", "b", "c", "b", "c", "a"], 2)))
print(repr(kth_distinct_sets(["aaa", "aa", "a"], 1)))
print(repr(kth_distinct_sets(["a", "b", "a"], 3)))

'a'
'aaa'
''


Ưu điểm nhỏ: lượt 2 chỉ duyệt các **giá trị khác nhau** (≤ n), không duyệt lại cả mảng.

## 8. Code nộp LeetCode

In [14]:
from typing import List

class Solution:
    def kthDistinct(self, arr: List[str], k: int) -> str:
        freq = {}
        for x in arr:
            freq[x] = freq.get(x, 0) + 1
        for x in arr:
            if freq[x] == 1:
                k -= 1
                if k == 0:
                    return x
        return ""

sol = Solution()
print(repr(sol.kthDistinct(["d", "b", "c", "b", "c", "a"], 2)))

'a'


Phiên bản dùng `Counter` (ngắn hơn, cũng nộp được):

In [15]:
class SolutionCounter:
    def kthDistinct(self, arr: List[str], k: int) -> str:
        freq = Counter(arr)
        distinct = [x for x in arr if freq[x] == 1]
        return distinct[k - 1] if k <= len(distinct) else ""

print(repr(SolutionCounter().kthDistinct(["a", "b", "a"], 3)))

''


## 9. Vì sao đúng?

- Sau lượt 1, `freq[x]` = **số lần chính xác** `x` xuất hiện trong `arr` (mỗi phần tử cộng đúng 1).
- Lượt 2 duyệt `arr` **theo thứ tự gốc**. **Invariant:** trước khi xét vị trí `i`, `k_ban_đầu − k` = số chuỗi distinct nằm ở các vị trí `< i`.
- Một chuỗi distinct chỉ xuất hiện **một** lần, nên nó được đếm đúng một lần — không bị đếm trùng.
- Khi `k` về 0 tại vị trí `i`, `arr[i]` là chuỗi distinct thứ `k_ban_đầu` → trả đúng.
- Nếu duyệt hết mà `k > 0` → số chuỗi distinct < `k_ban_đầu` → trả `""` đúng yêu cầu.

## 10. Độ phức tạp

Gọi `n = len(arr)`, `L` = độ dài tối đa của chuỗi (≤ 5). Băm một chuỗi tốn `O(L)`; vì `L ≤ 5` nên thường coi là hằng số.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. `arr.count` | `O(n² · L)` | `O(1)` | đếm lại nhiều lần |
| 2. Two-pass dict/Counter | `O(n · L)` trung bình | `O(n · L)` | lời giải chuẩn, dừng sớm |
| 3. List distinct | `O(n · L)` | `O(n · L)` | đọc giống đề, không dừng sớm |
| 4. seen/dup + dict thứ tự | `O(n · L)` | `O(n · L)` | lượt 2 chỉ duyệt giá trị khác nhau |

## 11. Bộ test

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<34} PASS ({actual!r})")

cases = [
    ("ví dụ 1", ["d", "b", "c", "b", "c", "a"], 2, "a"),
    ("ví dụ 2", ["aaa", "aa", "a"], 1, "aaa"),
    ("ví dụ 3 (không đủ k)", ["a", "b", "a"], 3, ""),
    ("1 phần tử, k=1", ["x"], 1, "x"),
    ("tất cả giống nhau", ["a", "a", "a"], 1, ""),
    ("distinct ở cuối", ["a", "a", "b"], 1, "b"),
    ("k = số distinct", ["p", "q", "p", "r"], 2, "r"),
    ("k vượt số distinct", ["p", "q", "p", "r"], 3, ""),
    ("xuất hiện 3 lần không distinct", ["a", "b", "a", "a", "c"], 2, "c"),
    ("chuỗi chứa nhau", ["ab", "a", "b", "ab"], 2, "b"),
]
fs = [kth_distinct_bruteforce, kth_distinct_two_pass, kth_distinct_list, kth_distinct_sets,
      Solution().kthDistinct, SolutionCounter().kthDistinct]
for name, arr, k, exp in cases:
    for f in fs:
        assert f(list(arr), k) == exp, (f, arr, k)
    check(name, Solution().kthDistinct(arr, k), exp)

ví dụ 1                            PASS ('a')
ví dụ 2                            PASS ('aaa')
ví dụ 3 (không đủ k)               PASS ('')
1 phần tử, k=1                     PASS ('x')
tất cả giống nhau                  PASS ('')
distinct ở cuối                    PASS ('b')
k = số distinct                    PASS ('r')
k vượt số distinct                 PASS ('')
xuất hiện 3 lần không distinct     PASS ('c')
chuỗi chứa nhau                    PASS ('b')


In [17]:
import random
random.seed(2053)
for _ in range(2000):
    n = random.randint(1, 10)
    arr = [random.choice(["a", "b", "c", "d", "ab", "ba"]) for _ in range(n)]
    k = random.randint(1, n)
    answers = {f(list(arr), k) for f in fs}
    assert len(answers) == 1, (arr, k, answers)
print("2000 bộ ngẫu nhiên: 6 cài đặt cho cùng kết quả — PASS")

2000 bộ ngẫu nhiên: 6 cài đặt cho cùng kết quả — PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: Hiểu "distinct" là "giá trị khác nhau" (dùng `set`)

In [18]:
def wrong_set_meaning(arr, k):
    uniq = list(dict.fromkeys(arr))     # các giá trị khác nhau, giữ thứ tự
    return uniq[k - 1] if k <= len(uniq) else ""

print("Sai :", repr(wrong_set_meaning(["d", "b", "c", "b", "c", "a"], 2)))   # 'b'
print("Đúng:", repr(kth_distinct_two_pass(["d", "b", "c", "b", "c", "a"], 2)))  # 'a'

Sai : 'b'
Đúng: 'a'


`dict.fromkeys(arr)` tạo dict có key là các phần tử của `arr` (giữ thứ tự, bỏ trùng) — một mẹo hay để "bỏ trùng có thứ tự", nhưng **không phải** điều bài này hỏi.

### Lỗi 2: Duyệt `set(arr)` hoặc `sorted(...)` ở lượt 2 — mất thứ tự gốc

`set` không giữ thứ tự; `sorted` đổi thành thứ tự bảng chữ cái. Đề yêu cầu **thứ tự xuất hiện**.

In [19]:
def wrong_sorted_order(arr, k):
    freq = Counter(arr)
    distinct = [x for x in sorted(freq) if freq[x] == 1]
    return distinct[k - 1] if k <= len(distinct) else ""

print("Sai :", repr(wrong_sorted_order(["d", "b", "c", "b", "c", "a"], 1)))   # 'a' (thứ tự abc)
print("Đúng:", repr(kth_distinct_two_pass(["d", "b", "c", "b", "c", "a"], 1)))  # 'd'

Sai : 'a'
Đúng: 'd'


### Lỗi 3: Lệch chỉ số — dùng `distinct[k]` thay cho `distinct[k - 1]`

Vừa trả sai phần tử, vừa có thể `IndexError` khi `k == len(distinct)`.

### Lỗi 4: Quên trường hợp không đủ `k` → `IndexError`

`distinct[k - 1]` khi `k > len(distinct)` sẽ lỗi. Phải kiểm tra `k <= len(distinct)`.

In [20]:
distinct = ["d", "a"]
for k in [1, 2, 3]:
    try:
        print(f"k={k}: distinct[k] =", distinct[k])
    except IndexError as e:
        print(f"k={k}: distinct[k] -> IndexError: {e}")
print("Đúng: distinct[k-1] với k=2 ->", distinct[2 - 1])

k=1: distinct[k] = a
k=2: distinct[k] -> IndexError: list index out of range
k=3: distinct[k] -> IndexError: list index out of range
Đúng: distinct[k-1] với k=2 -> a


### Lỗi 5: Đánh dấu distinct ngay khi gặp lần đầu (một lượt "ngây thơ")

Khi gặp `x` lần đầu, ta **chưa biết** nó có xuất hiện lại phía sau không. Phải đếm hết trước (hai lượt), hoặc xử lý thứ tự cẩn thận như cách 4.

In [21]:
def wrong_one_pass(arr, k):
    seen = set()
    for x in arr:
        if x not in seen:          # coi lần đầu gặp là distinct -> sai
            seen.add(x)
            k -= 1
            if k == 0:
                return x
    return ""

print("Sai :", repr(wrong_one_pass(["d", "b", "c", "b", "c", "a"], 2)))  # 'b'
print("Đúng:", repr(kth_distinct_two_pass(["d", "b", "c", "b", "c", "a"], 2)))

Sai : 'b'
Đúng: 'a'


## 13. Mẹo nhận dạng pattern

- "Phần tử xuất hiện **đúng một lần** / **unique** / **không lặp**" → **đếm tần suất** (dict/Counter) rồi lọc `== 1`.
- "Thứ k **theo thứ tự xuất hiện**" → lượt 2 duyệt **mảng gốc** (hoặc dict giữ thứ tự chèn), không duyệt `set`.
- Tần suất không đổi trong bài → **đếm một lần**, đừng gọi `count` trong vòng lặp.
- Bài cùng họ: 387 First Unique Character in a String (k = 1 trên chuỗi), 1748 Sum of Unique Elements, 2085 Count Common Words With One Occurrence, 451 Sort Characters By Frequency.

## 14. Biến thể

1. **k = 1 trên một chuỗi ký tự:** trả **chỉ số** ký tự không lặp đầu tiên (LeetCode 387).
2. **Thứ k từ cuối:** duyệt `reversed(arr)` ở lượt 2.
3. **Xuất hiện đúng `t` lần** thay vì 1: đổi `== 1` thành `== t`; lưu ý khi `t > 1` một chuỗi xuất hiện nhiều lần, nên phải duyệt **dict theo thứ tự lần đầu** để không đếm trùng.

In [22]:
def first_uniq_char(s):
    freq = Counter(s)
    for i, ch in enumerate(s):
        if freq[ch] == 1:
            return i
    return -1

def kth_with_frequency(arr, k, t):
    freq = Counter(arr)                 # Counter giữ thứ tự gặp lần đầu
    for x in freq:
        if freq[x] == t:
            k -= 1
            if k == 0:
                return x
    return ""

print(first_uniq_char("leetcode"), first_uniq_char("aabb"))
print(repr(kth_with_frequency(["d", "b", "c", "b", "c", "a"], 2, 2)))   # 'c'

0 -1
'c'


## 15. Bài tập tự luyện

**Bài 1 — Tính tay:** `arr = ["x","y","x","z","w","z"]`. Liệt kê các chuỗi distinct theo thứ tự; trả lời với `k = 1, 2, 3`.

**Bài 2 — Đoán output:** `Counter("aab")`, `list(dict.fromkeys("abca"))`, `Counter(["a"])["b"]`.

**Bài 3 — TODO:** viết `exercise_kth_distinct(arr, k)` bằng hai lượt với `dict.get` (không dùng Counter).

**Bài 4 — TODO:** `count_distinct(arr)` — trả **số** chuỗi distinct.

**Bài 5 — TODO biến thể:** `kth_distinct_from_end(arr, k)` — chuỗi distinct thứ `k` tính **từ cuối mảng**.

**Bài 6 — Mini challenge:** `kth_distinct_queries(arr, ks)` trả list đáp án cho **nhiều** giá trị `k` (vd 10⁴ truy vấn) — mỗi truy vấn phải `O(1)` sau khi tiền xử lý.

In [23]:
def exercise_kth_distinct(arr, k):
    # TODO 1: đếm freq bằng dict.get
    # TODO 2: duyệt arr theo thứ tự, gặp freq == 1 thì giảm k
    # TODO 3: k == 0 thì trả; hết mảng trả ""
    pass

# assert exercise_kth_distinct(["d","b","c","b","c","a"], 2) == "a"

In [24]:
def count_distinct(arr):
    # TODO
    pass

# assert count_distinct(["d","b","c","b","c","a"]) == 2

In [25]:
def kth_distinct_from_end(arr, k):
    # TODO
    pass

# assert kth_distinct_from_end(["d","b","c","b","c","a"], 1) == "a"
# assert kth_distinct_from_end(["d","b","c","b","c","a"], 2) == "d"

In [26]:
def kth_distinct_queries(arr, ks):
    # TODO: tiền xử lý list distinct một lần, mỗi truy vấn tra O(1)
    pass

# assert kth_distinct_queries(["d","b","c","b","c","a"], [1, 2, 3]) == ["d", "a", ""]

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | đếm từng chuỗi | x:2, y:1, z:2, w:1 | distinct: `y`, `w` → k=1 `y`, k=2 `w`, k=3 `""` |
| 2 | Counter đếm; fromkeys bỏ trùng có thứ tự | key thiếu trong Counter trả 0 | `Counter({'a':2,'b':1})`, `['a','b','c']`, `0` |
| 3 | mục 4 | `freq[x] = freq.get(x, 0) + 1` | lượt 2 như `kth_distinct_two_pass` |
| 4 | lọc `== 1` | `sum(1 for v in freq.values() if v == 1)` | dùng Counter |
| 5 | thứ tự ngược | lượt 2 duyệt `reversed(arr)` | phần còn lại giữ nguyên |
| 6 | tách tiền xử lý và truy vấn | `distinct = [...]` một lần | `distinct[k-1] if k <= len(distinct) else ""` |

## 17. Đáp án tham khảo

**Bài 1:** freq = `{x:2, y:1, z:2, w:1}` → distinct theo thứ tự: `y`, `w`. k=1 → `"y"`, k=2 → `"w"`, k=3 → `""`.

**Bài 2:** `Counter({'a': 2, 'b': 1})` · `['a', 'b', 'c']` · `0`.

In [27]:
arr = ["x", "y", "x", "z", "w", "z"]
print([repr(kth_distinct_two_pass(arr, k)) for k in (1, 2, 3)])
print(Counter("aab"), list(dict.fromkeys("abca")), Counter(["a"])["b"])

["'y'", "'w'", "''"]
Counter({'a': 2, 'b': 1}) ['a', 'b', 'c'] 0


In [28]:
def exercise_kth_distinct_answer(arr, k):
    freq = {}
    for x in arr:
        freq[x] = freq.get(x, 0) + 1
    for x in arr:
        if freq[x] == 1:
            k -= 1
            if k == 0:
                return x
    return ""

def count_distinct_answer(arr):
    return sum(1 for v in Counter(arr).values() if v == 1)

def kth_distinct_from_end_answer(arr, k):
    freq = Counter(arr)
    for x in reversed(arr):
        if freq[x] == 1:
            k -= 1
            if k == 0:
                return x
    return ""

def kth_distinct_queries_answer(arr, ks):
    freq = Counter(arr)
    distinct = [x for x in arr if freq[x] == 1]          # tiền xử lý O(n)
    return [distinct[k - 1] if 1 <= k <= len(distinct) else "" for k in ks]   # O(1) mỗi truy vấn

assert exercise_kth_distinct_answer(["d", "b", "c", "b", "c", "a"], 2) == "a"
assert count_distinct_answer(["d", "b", "c", "b", "c", "a"]) == 2
assert kth_distinct_from_end_answer(["d", "b", "c", "b", "c", "a"], 1) == "a"
assert kth_distinct_from_end_answer(["d", "b", "c", "b", "c", "a"], 2) == "d"
assert kth_distinct_queries_answer(["d", "b", "c", "b", "c", "a"], [1, 2, 3]) == ["d", "a", ""]
random.seed(8)
for _ in range(500):
    arr = [random.choice("abcde") for _ in range(random.randint(1, 9))]
    ks = list(range(1, len(arr) + 1))
    assert kth_distinct_queries_answer(arr, ks) == [kth_distinct_bruteforce(arr, k) for k in ks]
    assert [exercise_kth_distinct_answer(arr, k) for k in ks] == [kth_distinct_bruteforce(arr, k) for k in ks]
print("Tất cả đáp án bài tập — PASS")

Tất cả đáp án bài tập — PASS


## 18. Tổng kết — template + checklist

**Template two-pass counting:**

```python
freq = {}
for x in arr:                          # lượt 1: đếm
    freq[x] = freq.get(x, 0) + 1
for x in arr:                          # lượt 2: duyệt theo thứ tự gốc
    if freq[x] == 1:
        k -= 1
        if k == 0:
            return x
return ""                              # không đủ k
```

**Checklist submit**

- [ ] "Distinct" = xuất hiện **đúng 1 lần**, không phải "giá trị khác nhau".
- [ ] Lượt 2 duyệt **mảng gốc** (giữ thứ tự), không duyệt `set`/`sorted`.
- [ ] `k` là 1-indexed (`distinct[k-1]`).
- [ ] Trả `""` khi không đủ `k` chuỗi distinct (không để `IndexError`).
- [ ] Tên method `kthDistinct`, trả `str`.